In [4]:
# Cell 1: Install & Imports + Load/Split
!pip install mlflow pytest scikit-learn imbalanced-learn  # Add imblearn for future SMOTE

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')


# Load final from Task 4
df_final = pd.read_csv('../data/processed/xente_final_with_target.csv')
print(f"Final data shape: {df_final.shape}")
print("Target balance:", df_final['is_high_risk'].value_counts(normalize=True).round(3))

# Feats: Exclude IDs/target
id_cols = ['CustomerId', 'TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'TransactionStartTime']
categorical_cols = ['CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId']  # From your code
feat_cols = [col for col in df_final.columns if col not in id_cols + ['is_high_risk']]
num_cols = [col for col in feat_cols if col not in categorical_cols]  # Num: Amount, Value, time, etc.

X = df_final[feat_cols]
y = df_final['is_high_risk']

# Explicitly convert categorical columns to 'category' dtype
for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].astype('category')

# Split 80/20, stratify
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train shape: {X_train.shape}, Test: {X_test.shape}")

# Cell 2: Preprocessor + Models in Pipeline (Fixes String Error)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), num_cols)
    ],
    remainder='drop'  # Drop any extras
)

# Pipelines
lr_pipeline = Pipeline([('preproc', preprocessor), ('clf', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))])  # Balanced for imbalance
rf_pipeline = Pipeline([('preproc', preprocessor), ('clf', RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'))])

# Start MLflow
mlflow.set_experiment("credit_risk_modeling")
with mlflow.start_run(run_name="baseline_models"):
    # Tune LR
    lr_params = {'clf__C': [0.1, 1, 10]}
    lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=5, scoring='roc_auc')
    lr_grid.fit(X_train, y_train)
    lr_best = lr_grid.best_estimator_
    mlflow.log_params(lr_grid.best_params_)
    mlflow.sklearn.log_model(lr_best, "logistic_model")
    mlflow.log_metric("lr_auc", lr_grid.best_score_)

    # Tune RF
    rf_params = {'clf__n_estimators': [50, 100], 'clf__max_depth': [5, 10]}
    rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring='roc_auc')
    rf_grid.fit(X_train, y_train)
    rf_best = rf_grid.best_estimator_
    mlflow.log_params(rf_grid.best_params_)
    mlflow.sklearn.log_model(rf_best, "rf_model")
    mlflow.log_metric("rf_auc", rf_grid.best_score_)

print("Models trained/tuned, logged to MLflow. Best LR C:", lr_grid.best_params_['clf__C'])
print("Best RF params:", rf_grid.best_params_)

# Cell 3: Evaluation
lr_pred = lr_best.predict(X_test)
lr_proba = lr_best.predict_proba(X_test)[:, 1]
rf_pred = rf_best.predict(X_test)
rf_proba = rf_best.predict_proba(X_test)[:, 1]

def eval_metrics(y_true, y_pred, y_proba):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

lr_metrics = eval_metrics(y_test, lr_pred, lr_proba)
rf_metrics = eval_metrics(y_test, rf_pred, rf_proba)

comparison = pd.DataFrame({'Logistic Regression': lr_metrics, 'Random Forest': rf_metrics}).T
print("Model Comparison:\n", comparison.round(4))

best_model = 'RF' if rf_metrics['ROC-AUC'] > lr_metrics['ROC-AUC'] else 'LR'
best_model_obj = rf_best if best_model == 'RF' else lr_best
print(f"\nBest model: {best_model} (AUC: {max(rf_metrics['ROC-AUC'], lr_metrics['ROC-AUC']):.4f})")

if best_model == 'RF':
    print("\nRF Report:\n", classification_report(y_test, rf_pred))
else:
    print("\nLR Report:\n", classification_report(y_test, lr_pred))

# Cell 4: Register Best
with mlflow.start_run(run_name="best_model_registry"):
    mlflow.sklearn.log_model(best_model_obj, "best_model")
    for k, v in eval_metrics(y_test, best_model_obj.predict(X_test), best_model_obj.predict_proba(X_test)[:, 1]).items():
        mlflow.log_metric(f"best_{k.lower()}", v)
    mlflow.sklearn.log_model(best_model_obj, "production_model", registered_model_name="CreditRiskModel")
mlflow.end_run()
print("Best model registered in MLflow Registry as 'CreditRiskModel'.")

# Cell 5: Unit Tests
def test_model_training():
    assert len(feat_cols) > 0, "No features selected"
    assert lr_metrics['ROC-AUC'] > 0.5, "LR AUC below baseline"
    assert rf_metrics['ROC-AUC'] > 0.5, "RF AUC below baseline"

def test_target_balance():
    assert abs(y.mean() - 0.379) < 0.01, "Target balance mismatch"

test_model_training()
test_target_balance()
print("Unit tests passed!")

# Export tests to file
test_code = '''
import pytest
import pandas as pd
from sklearn.metrics import roc_auc_score

def test_model_auc(df_final):
    y = df_final["is_high_risk"]
    # Mock pred_proba
    pred_proba = np.random.rand(len(y))
    auc = roc_auc_score(y, pred_proba)
    assert auc > 0.5, "Mock AUC below baseline"

def test_feature_count(df_final):
    feat_cols = [col for col in df_final.columns if col not in ["CustomerId", "is_high_risk"]]
    assert len(feat_cols) >= 10, "Too few features"
'''
with open('../test/test_model_training.py', 'w') as f:
    f.write(test_code)
print("Tests exported to test_model_training.py—run pytest locally.")

Final data shape: (3742, 22)
Target balance: is_high_risk
0    0.621
1    0.379
Name: proportion, dtype: float64
Train shape: (2993, 15), Test: (749, 15)
Models trained/tuned, logged to MLflow. Best LR C: 10
Best RF params: {'clf__max_depth': 10, 'clf__n_estimators': 100}
Model Comparison:
                      Accuracy  Precision  Recall      F1  ROC-AUC
Logistic Regression    0.7343     0.6049  0.8627  0.7112   0.8049
Random Forest          0.7543     0.6202  0.9085  0.7371   0.8644

Best model: RF (AUC: 0.8644)

RF Report:
               precision    recall  f1-score   support

           0       0.92      0.66      0.77       465
           1       0.62      0.91      0.74       284

    accuracy                           0.75       749
   macro avg       0.77      0.78      0.75       749
weighted avg       0.81      0.75      0.76       749

Best model registered in MLflow Registry as 'CreditRiskModel'.
Unit tests passed!
Tests exported to test_model_training.py—run pytest locall

Registered model 'CreditRiskModel' already exists. Creating a new version of this model...
Created version '4' of model 'CreditRiskModel'.
